# LangGraph Tutorials

This notebook demonstrates:
1. A simple linear LangGraph workflow
2. A validation/self-correction loop
3. A ReAct-style agent with tool calling
4. Streaming graph execution

The code uses the current LangGraph message-state pattern.


In [ ]:
# Install required packages
# !pip install -U langgraph langchain langchain-openai python-dotenv -q

### 2. Set OpenAI API Key

Create a `.env` file in the same folder as this notebook:

```text
OPENAI_API_KEY=your_api_key_here
```

Do not hard-code your API key in the notebook.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add it to your .env file "
        "or set it in your environment before running the notebook."
    )


### 3. Imports & LLM Setup

### 4. Define State and Nodes

`StateGraph` requires a state schema. `add_messages` is used as the reducer so nodes can return only the new messages instead of manually rebuilding the entire message history.


### 5. Simple Linear Graph

The graph is:

```text
START → LLM → Validator → END
```


### 6. Graph with Loop (Self-Correction)

This version demonstrates conditional routing:

```text
                 ┌──────────────┐
                 │              │
START → LLM → Validator ────────┘
                 │
                 └────────────→ END
```

If validation fails, execution goes back to the LLM.


### 7. ReAct Agent

The important part of a tool-calling graph is:

```text
START → Agent → Tools → Agent → ... → END
```

When the model requests a tool, `ToolNode` executes it and creates the required `ToolMessage`. This prevents the OpenAI error that occurs when an assistant `tool_call` is not followed by a matching tool response.


### 8. Test the ReAct Agent

In [ ]:
final_result = react_app.invoke({
    "messages": [
        HumanMessage(content="Search for what is LangGraph")
    ],
    "error": None,
    "next": "",
})

print("Final Answer:")
print(final_result["messages"][-1].content)


### 9. Stream Output

`stream_mode="values"` shows the state after each graph step. Tool calls may have empty textual content, so the helper below prints the message type and useful content.
